# 作业二：Benchmark · VS Code + Colab

1. 在 **VS Code** 打开本 notebook，右上角 **Select Kernel → Colab → New Colab Server**，选择 **A100 ≥40GB、高 RAM**。
2. 运行第一格，点击 **上传代码包**，选择本地 `assignment2_colab.zip`；按钮显示 **(1)** 后，再运行后面的代码格。
3. 结果自动保存到 Drive 的 `MyDrive/cs336/assignment2/`。

复用所选内核的 Python 和 CUDA PyTorch。只做 `benchmarking_script (a)–(c)`：`batch=4`、`context=128`、FP32。


## 1. 点击按钮上传代码包
先完成文件选择，再运行下一格。


## 2. 解压代码，准备环境
挂载 Drive 保存结果；实际 Python、PyTorch 和显卡信息保存到 `environment.json`。


In [ ]:
from google.colab import drive
from pathlib import Path
import io, json, os, subprocess, sys, zipfile
from datetime import datetime

drive.mount('/content/drive')
PROJECT = Path('/content/assignment2-systems')
with zipfile.ZipFile('/content/drive/MyDrive/cs336/assignment2_colab.zip') as archive:
    archive.extractall(PROJECT)
os.chdir(PROJECT)
os.environ['PYTHONPATH'] = str(PROJECT / 'cs336-basics')
RESULTS = Path('/content/drive/MyDrive/cs336/assignment2') / datetime.now().strftime('%Y%m%d_%H%M%S')
RESULTS.mkdir(parents=True)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'einops==0.8.1', 'einx==0.3.0', 'jaxtyping==0.3.2', 'pandas'], check=True,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
     '--ignore-requires-python', '-e', str(PROJECT)], check=True,
)
import torch
RUN = [sys.executable]
environment = dict(
    python=sys.version.split()[0], torch=torch.__version__, cuda=torch.version.cuda,
    gpu=torch.cuda.get_device_name(0),
    vram_gib=torch.cuda.get_device_properties(0).total_memory / 2**30,
)
(RESULTS / 'environment.json').write_text(json.dumps(environment, indent=2))
print(environment)
print('结果目录:', RESULTS)

In [9]:
%pip install -q --no-deps --ignore-requires-python -e /content/assignment2-systems/

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for cs336-systems (pyproject.toml) ... done


## 3. 实验参数
(b) 使用预热 5 次；(c) 比较 0、1、2 次。默认共 20 组，每组测量 10 次。

In [10]:
MODELS = ['small', 'medium', 'large', 'xl', '2.7B']
WARMUPS = [5, 0, 1, 2]
CONTEXT_LENGTH = 128
MEASUREMENT_STEPS = 10

## 4. 批量运行
每组启动独立进程，JSON 和日志直接保存到 Drive；失败时显示日志并停止。

In [11]:
for model in MODELS:
    for warmup in WARMUPS:
        name = f'{model}_w{warmup}'
        command = RUN + ['-m', 'cs336_systems.benchmark',
            '--model-size', model, '--context-length', str(CONTEXT_LENGTH),
            '--batch-size', '4', '--vocab-size', '10000',
            '--device', 'cuda', '--dtype', 'float32', '--mode', 'forward-backward',
            '--warmup-steps', str(warmup), '--measurement-steps', str(MEASUREMENT_STEPS),
            '--output', str(RESULTS / f'{name}.json')]
        result = subprocess.run(command, capture_output=True, text=True)
        (RESULTS / f'{name}.log').write_text(result.stdout + result.stderr)
        if result.returncode:
            print(result.stdout + result.stderr)
            result.check_returncode()
        print(name, '完成', flush=True)

medium_w1 完成
medium_w2 完成
large_w5 完成
large_w0 完成
large_w1 完成
large_w2 完成
xl_w5 完成
xl_w0 完成
xl_w1 完成
xl_w2 完成
2.7B_w5 完成
2.7B_w0 完成
2.7B_w1 完成
2.7B_w2 完成


In [ ]:
print(result.stdout)
print(result.stderr)

## 5. 汇总结果
单位为毫秒。表格保存为 `summary.csv`，逐次测量在各组 JSON 的 `measurements` 中。

In [12]:
import pandas as pd

rows = []
for model in MODELS:
    for warmup in WARMUPS:
        data = json.loads((RESULTS / f'{model}_w{warmup}.json').read_text())
        row = {'model': model, 'context': data['config']['context_length'], 'warmup': warmup}
        for stage, timing in data['summary'].items():
            row[f'{stage}_ms'] = timing['mean_seconds'] * 1000
            row[f'{stage}_std_ms'] = timing['std_seconds'] * 1000
        rows.append(row)
summary = pd.DataFrame(rows)
summary.to_csv(RESULTS / 'summary.csv', index=False)
display(summary.round(3))

,model,context,warmup,forward_ms,forward_std_ms,loss_ms,loss_std_ms,backward_ms,backward_std_ms
0,small,128,5,13.465,0.165,0.090,0.001,14.293,0.091
1,small,128,0,46.174,103.588,1.325,3.894,20.804,19.767
2,small,128,1,13.325,0.418,0.092,0.005,13.637,0.638
3,small,128,2,13.469,0.126,0.090,0.001,14.435,0.043
4,medium,128,5,26.643,0.185,0.093,0.003,32.658,0.021
5,medium,128,0,58.155,99.133,0.927,2.627,38.585,18.725
6,medium,128,1,26.218,0.220,0.094,0.007,32.659,0.085
7,medium,128,2,26.307,0.128,0.094,0.004,32.678,0.030
8,large,128,5,39.734,0.420,0.091,0.002,67.776,0.450
9,large,128,0,62.723,70.972,0.934,2.654,73.368,16.864
